In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_path = '/content/drive/MyDrive/TyreNet'

STEP 2: Load Images and Labels

In [4]:
IMG_SIZE = 128
def load_data(base_path):
    images = []
    labels = []

    categories = ["Defective", "Good"]

    for category in categories:
        path = os.path.join(base_path, category)

        for file in os.listdir(path):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)

            # 🔥 FINAL LABEL SYSTEM
            if category == "Good":
                labels.append(0.9)
            else:
                labels.append(0.3)

    images = np.array(images) / 255.0
    labels = np.array(labels)

    return images, labels


images, labels = load_data(data_path)

print("Images:", images.shape)
print("Labels:", labels.shape)


Images: (1698, 128, 128, 3)
Labels: (1698,)


STEP 3: Generate Masks Automatically

In [5]:
def create_mask(img):
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0
    mask = np.expand_dims(mask, axis=-1)

    return mask

masks = np.array([create_mask(img) for img in images])

print("Masks:", masks.shape)

Masks: (1698, 128, 128, 1)


train test split

In [6]:
X_train, X_test, M_train, M_test, y_train, y_test = train_test_split(
    images, masks, labels, test_size=0.2, random_state=42
)

STEP 4: Build UNet for Segmentation

In [7]:
def dice_loss(y_true, y_pred):
    smooth = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - ((2. * intersection + smooth) /
                (K.sum(y_true_f) + K.sum(y_pred_f) + smooth))

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(input_shape)

    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D()(c2)

    b = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)

    u1 = layers.UpSampling2D()(b)
    u1 = layers.concatenate([u1, c2])
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.concatenate([u2, c1])
    c4 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c4)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss=lambda y_true, y_pred: dice_loss(y_true, y_pred) +
                                   tf.keras.losses.binary_crossentropy(y_true, y_pred)
    )

    return model

unet = build_unet()
unet.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 64,    │          0 │ conv2d_2[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 64,    │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 192)              │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    110,656 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 128,  │          0 │ conv2d_3[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, 128,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 96)               │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │     27,680 │ concatenate_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 128, 128,  │         33 │ conv2d_4[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 231,617 (904.75 KB)

 Trainable params: 231,617 (904.75 KB)

 Non-trainable params: 0 (0.00 B)

STEP 5: Train UNet

In [16]:
unet.fit(
    X_train, M_train,
    validation_data=(X_test, M_test),
    epochs=50,
    batch_size=4
)

Epoch 1/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.4492 - val_loss: 0.4741
Epoch 2/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4493 - val_loss: 0.4725
Epoch 3/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4472 - val_loss: 0.4740
Epoch 4/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4469 - val_loss: 0.4778
Epoch 5/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4441 - val_loss: 0.4725
Epoch 6/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4441 - val_loss: 0.4793
Epoch 7/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4422 - val_loss: 0.4748
Epoch 8/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4392 - val_loss: 0.4718
Epoch 9/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4404 - val_loss: 0.4705
Epoch 10/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4380 - val_loss: 0.4731
Epoch 11/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4363 - val_loss: 0.4806
Epoch 12/50
340/340 ━━━━━━━━━━━━━━━━━━━━ 

STEP 6: Prepare Masks for CNN Regression

In [17]:
def get_clean_mask(model, images):
    preds = model.predict(images)
    return (preds > 0.5).astype(np.float32)

train_masks = get_clean_mask(unet, X_train)
test_masks = get_clean_mask(unet, X_test)

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


STEP 7: Build CNN Regression with Attention

In [18]:
def attention_block(x):
    attention = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return layers.Multiply()([x, attention])

def build_cnn():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 1))

    x = layers.Conv2D(32, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = attention_block(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.Huber(),
        metrics=['mae']
    )

    return model

cnn = build_cnn()
cnn.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 128, 128,  │        320 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 128, 128,  │         33 │ activation_3[0][… │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 128, 128,  │          0 │ activation_3[0][… │
│ (Multiply)          │ 32)               │            │ conv2d_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 64,    │          0 │ multiply_1[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 64, 64,    │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 32, 32,    │          0 │ activation_4[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 32, 32,    │     73,856 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_13[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ activation_5[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 101,922 (398.13 KB)

 Trainable params: 101,474 (396.38 KB)

 Non-trainable params: 448 (1.75 KB)

STEP 8: Map Labels to Tyre Life (%)

STEP 9: Train CNN Regression

In [19]:
cnn.fit(
    train_masks, y_train,
    validation_data=(test_masks, y_test),
    epochs=30,
    batch_size=4
)

Epoch 1/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - loss: 0.0688 - mae: 0.2985 - val_loss: 0.0395 - val_mae: 0.2468
Epoch 2/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0403 - mae: 0.2303 - val_loss: 0.0302 - val_mae: 0.2119
Epoch 3/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0353 - mae: 0.2180 - val_loss: 0.0314 - val_mae: 0.1996
Epoch 4/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0355 - mae: 0.2182 - val_loss: 0.0327 - val_mae: 0.1798
Epoch 5/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0334 - mae: 0.2112 - val_loss: 0.0314 - val_mae: 0.2037
Epoch 6/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0326 - mae: 0.2063 - val_loss: 0.0719 - val_mae: 0.2910
Epoch 7/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0323 - mae: 0.2068 - val_loss: 0.0372 - val_mae: 0.1954
Epoch 8/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0317 - mae: 0.2034 - val_loss: 0.0350 - val_mae: 0.1914
Epoch 9/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - 

STEP 10: Test Prediction

In [20]:
def predict(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    mask = unet.predict(img)
    mask = (mask > 0.5).astype(np.float32)

    pred = cnn.predict(mask)[0][0]
    pred = np.clip(pred, 0, 1)

    life_percent = pred * 100
    wear_percent = 100 - life_percent
    remaining_km = pred * 40000

    return wear_percent, life_percent, remaining_km

test

In [21]:
wear, life, km = predict('/content/drive/MyDrive/dummytest/t1.jpg')

print(f"Wear: {wear:.2f}%")
print(f"Life: {life:.2f}%")
print(f"Remaining Distance: {km:.0f} km")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 466ms/step
Wear: 8.72%
Life: 91.28%
Remaining Distance: 36510 km


STEP 11: Save Models for Reuse Anywhere

In [ ]:
# Save UNet
unet_model.save('/content/drive/MyDrive/unet_tyre_model.h5')

# Save CNN regression model
cnn_reg_model.save('/content/drive/MyDrive/cnn_regression_attention.h5')